# MIMIC-IV Pre-target Item Exploration

Explore MIMIC item metadata before creating target datasets. This notebook owns the exploratory regex guesses for the current hypoglycemia, hypokalemia, and hypotension setup: after reviewing matches, copy selected variable -> item ID lists and regexes into `configs/datasets/mimic/targets.yaml` or the legacy `configs/mimic_targets.yaml`.


## Setup

In [4]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / 'src' / 'interpretable_ts_vit').exists():
        PROJECT_DIR = candidate
        break
else:
    raise FileNotFoundError(f'Could not find repository root from {cwd}')

SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
os.chdir(PROJECT_DIR)

CONFIG_PATH = PROJECT_DIR / 'configs' / 'datasets' / 'mimic' / 'targets.yaml'
print('PROJECT_DIR:', PROJECT_DIR)
print('CONFIG_PATH:', CONFIG_PATH)

PROJECT_DIR: C:\Users\micha\Documents\Explainable ViT for Clinical Data
CONFIG_PATH: C:\Users\micha\Documents\Explainable ViT for Clinical Data\configs\datasets\mimic\targets.yaml


In [5]:
import re
import pandas as pd

from interpretable_ts_vit.datasets import load_mimic_targets_config
from interpretable_ts_vit.datasets.mimic.mimic_iv import _MIMICSource

config = load_mimic_targets_config(CONFIG_PATH)
mimic_path = PROJECT_DIR / config.mimic_path if not Path(config.mimic_path).is_absolute() else Path(config.mimic_path)
cache_dir = PROJECT_DIR / config.cache_dir if config.cache_dir and not Path(config.cache_dir).is_absolute() else Path(config.cache_dir) if config.cache_dir else None
extraction_dir = cache_dir / 'dictionary_extracted' if cache_dir and config.use_extracted_files else None
source = _MIMICSource(mimic_path, extraction_dir=extraction_dir)

def match_patterns(text: pd.Series, patterns: list[str]) -> pd.Series:
    if not patterns:
        return pd.Series(False, index=text.index)
    combined = '|'.join(f'(?:{pattern})' for pattern in patterns)
    return text.str.contains(combined, case=False, regex=True, na=False)

print('MIMIC source:', mimic_path)
print('Dictionary extraction dir:', extraction_dir)
print('Configured targets:', config.targets)
print('Configured windows:', [(w.name, w.observation_hours, w.prediction_hours, w.gap_hours) for w in config.windows])
print('Target dataset creation is not run here.')


MIMIC source: C:\Users\micha\Documents\Explainable ViT for Clinical Data\mimic-iv-3.1.zip
Dictionary extraction dir: C:\Users\micha\Documents\Explainable ViT for Clinical Data\data\mimic_targets\cache\dictionary_extracted
Configured targets: ['hypoglycemia', 'hypokalemia', 'hypotension']
Configured windows: [('obs24_target8_gap0', 24, 8, 0), ('obs24_target8_gap2', 24, 8, 2)]
Target dataset creation is not run here.


In [6]:
# Exploratory regex guesses live in this notebook, not in the data creation module.
# Edit these freely while deciding which item IDs should become the explicit config.
EXPLORATORY_LAB_REGEXES = {
    'blood_glucose': [r'\bglucose\b'],
    'wbc': [r'white blood cells?', r'\bwbc\b'],
    'creatinine': [r'\bcreatinine\b'],
    'bun': [r'blood urea nitrogen', r'\bbun\b', r'urea nitrogen'],
    'potassium': [r'\bpotassium\b'],
    'magnesium': [r'\bmagnesium\b'],
    'hemoglobin': [r'\bhemoglobin\b'],
    'pco2': [r'\bpco2\b', r'carbon dioxide'],
    'platelet_count': [r'platelet'],
    'po2': [r'\bpo2\b', r'oxygen'],
    'sodium': [r'\bsodium\b'],
}

EXPLORATORY_CHART_ITEMIDS = {
    'heart_rate': [220045],
    'temperature': [223761, 223762],
    'systolic_bp': [220050, 220179],
    'diastolic_bp': [220051, 220180],
    'o2_saturation_pulseox': [220277],
}

EXPLORATORY_CHART_REGEXES = {
    'systolic_bp': [r'\bsystolic\b'],
    'diastolic_bp': [r'\bdiastolic\b'],
    'o2_saturation_pulseox': [r'o2 saturation', r'spo2', r'pulse ox'],
}

EXPLORATORY_INPUTEVENT_REGEXES = {
    'previous_dextrose_treatment': [r'dextrose'],
    'previous_insulin_treatment': [r'\binsulin\b'],
    'previous_potassium_chloride_treatment': [r'potassium chloride', r'\bkcl\b'],
    'previous_dopamine_treatment': [r'dopamine'],
    'previous_norepinephrine_treatment': [r'norepinephrine', r'noradrenaline'],
    'previous_fluids_given': [
        r'crystalloid', r'normal saline', r'lactated ringer', r'\blr\b',
        r'dextrose.*water', r'\bd5w\b', r'sterile water', r'\biv fluid',
    ],
}

EXPLORATORY_DRUG_REGEXES = {}


## Read Item Metadata Tables


In [7]:
lab_items = source.read_table('hosp/d_labitems.csv.gz', usecols=['itemid', 'label', 'fluid', 'category'])
icu_items = source.read_table('icu/d_items.csv.gz', usecols=['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname', 'param_type', 'lownormalvalue', 'highnormalvalue'])

lab_items['search_text'] = lab_items[['label', 'fluid', 'category']].fillna('').astype(str).agg(' '.join, axis=1).str.lower()
icu_items['search_text'] = icu_items[['label', 'abbreviation', 'category', 'unitname', 'param_type']].fillna('').astype(str).agg(' '.join, axis=1).str.lower()

print('d_labitems rows:', len(lab_items))
print('d_items rows:', len(icu_items))
display(lab_items.head())
display(icu_items.head())


d_labitems rows: 1650
d_items rows: 4095


,itemid,label,fluid,category,search_text
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas,alveolar-arterial gradient blood blood gas
1,50802,Base Excess,Blood,Blood Gas,base excess blood blood gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas,"calculated bicarbonate, whole blood blood bloo..."
3,50804,Calculated Total CO2,Blood,Blood Gas,calculated total co2 blood blood gas
4,50805,Carboxyhemoglobin,Blood,Blood Gas,carboxyhemoglobin blood blood gas


,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue,search_text
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN,problem list problem list general text
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN,icu admission date icu admission date adt dat...
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN,heart rate hr routine vital signs bpm numeric
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN,heart rate alarm - high hr alarm - high alarms...
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN,heart rate alarm - low hr alarm - low alarms b...


## Lab Item Regex Matches

Each variable below is shown as a separate dataframe of candidate `hosp/labevents.csv.gz` item IDs matched from `hosp/d_labitems.csv.gz`.


In [8]:
lab_match_frames = {}
for variable, patterns in EXPLORATORY_LAB_REGEXES.items():
    matched = lab_items[match_patterns(lab_items['search_text'], patterns)].copy()
    exact_ids = {int(itemid) for itemid in config.lab_itemids.get(variable, [])}
    if exact_ids:
        matched = pd.concat([matched, lab_items[lab_items['itemid'].isin(exact_ids)]], ignore_index=True).drop_duplicates('itemid')
    matched = matched.sort_values(['label', 'itemid']).drop(columns=['search_text'])
    matched.insert(0, 'variable', variable)
    matched['patterns'] = ', '.join(patterns)
    matched['exact_config_id'] = matched['itemid'].astype(int).isin(exact_ids)
    lab_match_frames[variable] = matched
    print(f'\n{variable}: {matched["itemid"].nunique()} lab item IDs')
    display(matched)

lab_matches = pd.concat(lab_match_frames.values(), ignore_index=True) if lab_match_frames else pd.DataFrame()
display(lab_matches.groupby('variable')['itemid'].nunique().rename('matched_lab_itemids').to_frame() if not lab_matches.empty else pd.DataFrame())



blood_glucose: 13 lab item IDs


,variable,itemid,label,fluid,category,patterns,exact_config_id
0,blood_glucose,50809,Glucose,Blood,Blood Gas,\bglucose\b,True
2,blood_glucose,50931,Glucose,Blood,Chemistry,\bglucose\b,True
7,blood_glucose,51478,Glucose,Urine,Hematology,\bglucose\b,False
10,blood_glucose,51981,Glucose,Urine,Chemistry,\bglucose\b,False
12,blood_glucose,52569,Glucose,Blood,Chemistry,\bglucose\b,False
1,blood_glucose,50842,"Glucose, Ascites",Ascites,Chemistry,\bglucose\b,False
4,blood_glucose,51034,"Glucose, Body Fluid",Other Body Fluid,Chemistry,\bglucose\b,False
8,blood_glucose,51790,"Glucose, CSF",Cerebrospinal Fluid,Chemistry,\bglucose\b,False
3,blood_glucose,51022,"Glucose, Joint Fluid",Joint Fluid,Chemistry,\bglucose\b,False
5,blood_glucose,51053,"Glucose, Pleural",Pleural,Chemistry,\bglucose\b,False



wbc: 8 lab item IDs


,variable,itemid,label,fluid,category,patterns,exact_config_id
2,wbc,51516,WBC,Urine,Hematology,"white blood cells?, \bwbc\b",False
7,wbc,52407,WBC,Stool,Hematology,"white blood cells?, \bwbc\b",False
3,wbc,51517,WBC Casts,Urine,Hematology,"white blood cells?, \bwbc\b",False
4,wbc,51518,WBC Clumps,Urine,Hematology,"white blood cells?, \bwbc\b",False
0,wbc,51300,WBC Count,Blood,Hematology,"white blood cells?, \bwbc\b",True
1,wbc,51301,White Blood Cells,Blood,Hematology,"white blood cells?, \bwbc\b",True
5,wbc,51755,White Blood Cells,Blood,Chemistry,"white blood cells?, \bwbc\b",False
6,wbc,51756,White Blood Cells,Blood,Chemistry,"white blood cells?, \bwbc\b",False



creatinine: 20 lab item IDs


...


po2: 5 lab item IDs


,variable,itemid,label,fluid,category,patterns,exact_config_id
0,po2,50816,Oxygen,Blood,Blood Gas,"\bpo2\b, oxygen",False
1,po2,50817,Oxygen Saturation,Blood,Blood Gas,"\bpo2\b, oxygen",False
2,po2,50821,pO2,Blood,Blood Gas,"\bpo2\b, oxygen",True
4,po2,52042,pO2,Fluid,Blood Gas,"\bpo2\b, oxygen",False
3,po2,50832,"pO2, Body Fluid",Other Body Fluid,Blood Gas,"\bpo2\b, oxygen",False



sodium: 13 lab item IDs


,variable,itemid,label,fluid,category,patterns,exact_config_id
3,sodium,50983,Sodium,Blood,Chemistry,\bsodium\b,True
12,sodium,52623,Sodium,Blood,Chemistry,\bsodium\b,False
2,sodium,50848,"Sodium, Ascites",Ascites,Chemistry,\bsodium\b,False
1,sodium,50834,"Sodium, Body Fluid",Other Body Fluid,Blood Gas,\bsodium\b,False
4,sodium,51042,"Sodium, Body Fluid",Other Body Fluid,Chemistry,\bsodium\b,False
8,sodium,51801,"Sodium, CSF",Cerebrospinal Fluid,Chemistry,\bsodium\b,False
9,sodium,51823,"Sodium, Joint Fluid",Joint Fluid,Chemistry,\bsodium\b,False
5,sodium,51058,"Sodium, Pleural",Pleural,Chemistry,\bsodium\b,False
6,sodium,51065,"Sodium, Stool",Stool,Chemistry,\bsodium\b,False
7,sodium,51100,"Sodium, Urine",Urine,Chemistry,\bsodium\b,False


,matched_lab_itemids
variable,
blood_glucose,13
bun,10
creatinine,20
hemoglobin,22
magnesium,3
pco2,3
platelet_count,8
po2,5
potassium,13


In [9]:
chosen_lab_itemids = {
    "blood_glucose": [50809, 50931, 52569, 52027],
    "wbc": [51300, 51301, 51755, 51756],
    "creatinine": [50912, 52546, 52024],
    "bun": [51006, 52647],
    "potassium": [50971, 52610, 50822, 52452],  
    "magnesium": [50960],
    "hemoglobin": [50811, 51222, 51640],
    "pco2": [50818],                            
    "platelet_count": [51265, 53189],
    "po2": [50821],                             
    "sodium": [50983, 52623, 50824, 52455],
}

## Chart Item IDs and Regex Matches

Each variable below is shown as a separate dataframe of exact `icu/chartevents.csv.gz` IDs plus regex candidates resolved against `icu/d_items.csv.gz`, including `unitname` and normal-range columns where MIMIC provides them.


In [10]:
chart_match_frames = {}
chart_variables = sorted(set(EXPLORATORY_CHART_ITEMIDS) | set(EXPLORATORY_CHART_REGEXES) | set(config.chart_itemids) | set(config.chart_regexes))
chart_display_columns = ['variable', 'itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname', 'param_type', 'lownormalvalue', 'highnormalvalue', 'patterns', 'exact_config_id', 'missing_from_dictionary']
for variable in chart_variables:
    requested = sorted({int(itemid) for itemid in EXPLORATORY_CHART_ITEMIDS.get(variable, [])} | {int(itemid) for itemid in config.chart_itemids.get(variable, [])})
    patterns = EXPLORATORY_CHART_REGEXES.get(variable, config.chart_regexes.get(variable, []))
    exact_ids = {int(itemid) for itemid in config.chart_itemids.get(variable, [])}
    matched = icu_items[icu_items['itemid'].isin(requested)].copy()
    if patterns:
        matched = pd.concat([matched, icu_items[match_patterns(icu_items['search_text'], patterns)]], ignore_index=True).drop_duplicates('itemid')
    missing = sorted(set(requested) - set(matched['itemid'].astype(int)))
    matched = matched.sort_values(['label', 'itemid']).drop(columns=['search_text'])
    matched.insert(0, 'variable', variable)
    matched['patterns'] = ', '.join(patterns)
    matched['exact_config_id'] = matched['itemid'].astype(int).isin(exact_ids)
    matched['missing_from_dictionary'] = False
    if missing:
        missing_frame = pd.DataFrame({'variable': variable, 'itemid': missing, 'missing_from_dictionary': True, 'patterns': ', '.join(patterns), 'exact_config_id': False})
        matched = pd.concat([matched, missing_frame], ignore_index=True)
    for column in chart_display_columns:
        if column not in matched.columns:
            matched[column] = pd.NA
    matched = matched[chart_display_columns]
    chart_match_frames[variable] = matched
    print(f'\n{variable}: {matched["itemid"].nunique()} chart item IDs/candidates')
    display(matched)

chart_matches = pd.concat(chart_match_frames.values(), ignore_index=True) if chart_match_frames else pd.DataFrame()
display(chart_matches.groupby('variable')['itemid'].nunique().rename('matched_chart_itemids').to_frame() if not chart_matches.empty else pd.DataFrame())



diastolic_bp: 11 chart item IDs/candidates


,variable,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue,patterns,exact_config_id,missing_from_dictionary
6,diastolic_bp,225310,ART BP Diastolic,ART BP Diastolic,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN,\bdiastolic\b,False,False
10,diastolic_bp,228151,Aortic Pressure Signal - Diastolic,Aortic Pressure Signal - Diastolic,chartevents,Impella,mmHg.,Numeric,NaN,NaN,\bdiastolic\b,False,False
0,diastolic_bp,220051,Arterial Blood Pressure diastolic,ABPd,chartevents,Routine Vital Signs,mmHg,Numeric,60.0,90.0,\bdiastolic\b,True,False
12,diastolic_bp,229900,Left Ventricular Pressure Signal - Diastolic,Left Ventricular Pressure Signal - Diastolic,chartevents,Impella,mmHg,Numeric,NaN,NaN,\bdiastolic\b,False,False
5,diastolic_bp,224643,Manual Blood Pressure Diastolic Left,Manual BPd L,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN,\bdiastolic\b,False,False
9,diastolic_bp,227242,Manual Blood Pressure Diastolic Right,Manual BPd R,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN,\bdiastolic\b,False,False
1,diastolic_bp,220180,Non Invasive Blood Pressure diastolic,NBPd,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN,\bdiastolic\b,True,False
8,diastolic_bp,226853,PA diastolic pressure(PA Line),PA diastolic pressure(PA Line),chartevents,PA Line Insertion,mmHg,Numeric,NaN,NaN,\bdiastolic\b,False,False
11,diastolic_bp,229668,Pulmonary Artery Pressure Signal - Diastolic,Pulmonary Artery Pressure Signal - Diastolic,chartevents,Impella,mmHg.,Numeric,NaN,NaN,\bdiastolic\b,False,False
3,diastolic_bp,220060,Pulmonary Artery Pressure diastolic,PAPd,chartevents,Hemodynamics,mmHg,Numeric,8.0,15.0,\bdiastolic\b,False,False



heart_rate: 1 chart item IDs/candidates


,variable,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue,patterns,exact_config_id,missing_from_dictionary
2,heart_rate,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN,,True,False



o2_saturation_pulseox: 11 chart item IDs/candidates


...


systolic_bp: 11 chart item IDs/candidates


,variable,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue,patterns,exact_config_id,missing_from_dictionary
6,systolic_bp,225309,ART BP Systolic,ART BP Systolic,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN,\bsystolic\b,False,False
10,systolic_bp,228152,Aortic Pressure Signal - Systolic,Aortic Pressure Signal - Systolic,chartevents,Impella,mmHg.,Numeric,NaN,NaN,\bsystolic\b,False,False
0,systolic_bp,220050,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0,\bsystolic\b,True,False
12,systolic_bp,229899,Left Ventricular Pressure Signal - Systolic,Left Ventricular Pressure Signal - Systolic,chartevents,Impella,mmHg,Numeric,NaN,NaN,\bsystolic\b,False,False
5,systolic_bp,224167,Manual Blood Pressure Systolic Left,Manual BPs L,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN,\bsystolic\b,False,False
9,systolic_bp,227243,Manual Blood Pressure Systolic Right,Manual BPs R,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN,\bsystolic\b,False,False
1,systolic_bp,220179,Non Invasive Blood Pressure systolic,NBPs,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN,\bsystolic\b,True,False
8,systolic_bp,226852,PA systolic pressure(PA Line),PA systolic pressure(PA Line),chartevents,PA Line Insertion,mmHg,Numeric,NaN,NaN,\bsystolic\b,False,False
3,systolic_bp,220059,Pulmonary Artery Pressure systolic,PAPs,chartevents,Hemodynamics,mmHg,Numeric,15.0,25.0,\bsystolic\b,False,False
11,systolic_bp,229669,Pulmonary Atrtery Pressure Signal - Systolic,Pulmonary Artery Pressure Signal - Systolic,chartevents,Impella,mmHg.,Numeric,NaN,NaN,\bsystolic\b,False,False



temperature: 2 chart item IDs/candidates


,variable,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue,patterns,exact_config_id,missing_from_dictionary
338,temperature,223762,Temperature Celsius,Temperature C,chartevents,Routine Vital Signs,°C,Numeric,NaN,NaN,,True,False
337,temperature,223761,Temperature Fahrenheit,Temperature F,chartevents,Routine Vital Signs,°F,Numeric,NaN,NaN,,True,False


,matched_chart_itemids
variable,
diastolic_bp,11
heart_rate,1
o2_saturation_pulseox,11
systolic_bp,11
temperature,2


In [12]:
chosen_chart_itemids = {
    "heart_rate": [
        220045,  # Heart Rate
    ],

    "systolic_bp": [
        220050,  # Arterial Blood Pressure systolic (invasive)
        225309,  # ART BP Systolic (same measurement, newer label)
        220179,  # Non Invasive Blood Pressure systolic
        224167,  # Manual Blood Pressure Systolic Left
        227243,  # Manual Blood Pressure Systolic Right
    ],

    "diastolic_bp": [
        220051,  # Arterial Blood Pressure diastolic (invasive)
        225310,  # ART BP Diastolic (same measurement, newer label)
        220180,  # Non Invasive Blood Pressure diastolic
        224643,  # Manual Blood Pressure Diastolic Left
        227242,  # Manual Blood Pressure Diastolic Right
    ],

    "o2_saturation_pulseox": [
        220277,  # O2 saturation pulseoxymetry (SpO2)
        220227,  # Arterial O2 Saturation (SaO2)
    ],

    "temperature_c": [
        223762,  # Temperature Celsius
    ],

    "temperature_f": [ 
        223761 # Temperature Fahrenheit
    ] 
}

## ICU Input Item Regex Matches

Each variable below is shown as a separate dataframe of candidate `icu/inputevents.csv.gz` item IDs matched from `icu/d_items.csv.gz`, including measurement units where MIMIC provides them.


In [13]:
input_match_frames = {}
input_display_columns = ['variable', 'itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname', 'param_type', 'lownormalvalue', 'highnormalvalue', 'patterns', 'exact_config_id']
for variable, patterns in EXPLORATORY_INPUTEVENT_REGEXES.items():
    matched = icu_items[match_patterns(icu_items['search_text'], patterns)].copy()
    exact_ids = {int(itemid) for itemid in config.inputevent_itemids.get(variable, [])}
    if exact_ids:
        matched = pd.concat([matched, icu_items[icu_items['itemid'].isin(exact_ids)]], ignore_index=True).drop_duplicates('itemid')
    matched = matched.sort_values(['label', 'itemid']).drop(columns=['search_text'])
    matched.insert(0, 'variable', variable)
    matched['patterns'] = ', '.join(patterns)
    matched['exact_config_id'] = matched['itemid'].astype(int).isin(exact_ids)
    for column in input_display_columns:
        if column not in matched.columns:
            matched[column] = pd.NA
    matched = matched[input_display_columns]
    input_match_frames[variable] = matched
    print(f'\n{variable}: {matched["itemid"].nunique()} inputevent item IDs')
    display(matched)

input_matches = pd.concat(input_match_frames.values(), ignore_index=True) if input_match_frames else pd.DataFrame()
display(input_matches.groupby('variable')['itemid'].nunique().rename('matched_input_itemids').to_frame() if not input_matches.empty else pd.DataFrame())



previous_dextrose_treatment: 18 inputevent item IDs


,variable,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue,patterns,exact_config_id
216,previous_dextrose_treatment,221000,Dextran 40 / Dextrose 5% (Gentran - Rheomacrodex),Dextran 40 / Dextrose 5%,inputevents,Fluids - Other (Not In Use),mL,Solution,NaN,NaN,dextrose,False
218,previous_dextrose_treatment,221002,Dextran 70 / Dextrose 5% (Gentran - Macrodex -...,Dextran 70 / Dextrose 5%,inputevents,Fluids - Other (Not In Use),mL,Solution,NaN,NaN,dextrose,False
170,previous_dextrose_treatment,220950,Dextrose 10%,Dextrose 10%,inputevents,Fluids/Intake,mL,Solution,NaN,NaN,dextrose,False
188,previous_dextrose_treatment,220968,Dextrose 10% / Ringers Lactate,Dextrose 10% / Ringers Lactate,inputevents,Fluids - Other (Not In Use),mL,Solution,NaN,NaN,dextrose,False
229,previous_dextrose_treatment,221014,Dextrose 19%,Dextrose 19%,inputevents,Fluids - Other (Not In Use),mL,Solution,NaN,NaN,dextrose,False
232,previous_dextrose_treatment,221017,"Dextrose 2,5%","Dextrose 2,5%",inputevents,Fluids - Other (Not In Use),mL,Solution,NaN,NaN,dextrose,False
186,previous_dextrose_treatment,220966,"Dextrose 2,5% / Saline 0,45%","Dextrose 2,5% / Saline 0,45%",inputevents,Fluids - Other (Not In Use),mL,Solution,NaN,NaN,dextrose,False
2686,previous_dextrose_treatment,228140,Dextrose 20%,Dextrose 20%,inputevents,Fluids/Intake,mL,Solution,NaN,NaN,dextrose,False
171,previous_dextrose_treatment,220951,Dextrose 20.%,Dextrose 20.%,inputevents,Fluids - Other (Not In Use),mL,Solution,NaN,NaN,dextrose,False
2687,previous_dextrose_treatment,228141,Dextrose 30%,Dextrose 30%,inputevents,Fluids/Intake,mL,Solution,NaN,NaN,dextrose,False



previous_insulin_treatment: 10 inputevent item IDs


,variable,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue,patterns,exact_config_id
323,previous_insulin_treatment,223257,Insulin - 70/30,Insulin - 70/30,inputevents,Medications,units,Solution,NaN,NaN,\binsulin\b,False
326,previous_insulin_treatment,223260,Insulin - Glargine,Insulin - Glargine,inputevents,Medications,units,Solution,NaN,NaN,\binsulin\b,False
328,previous_insulin_treatment,223262,Insulin - Humalog,Insulin - Humalog,inputevents,Medications,units,Solution,NaN,NaN,\binsulin\b,False
327,previous_insulin_treatment,223261,Insulin - Humalog 75/25,Insulin - Humalog 75/25,inputevents,Medications,units,Solution,NaN,NaN,\binsulin\b,False
325,previous_insulin_treatment,223259,Insulin - NPH,Insulin - NPH,inputevents,Medications,units,Solution,NaN,NaN,\binsulin\b,False
3517,previous_insulin_treatment,229299,Insulin - Novolog,Insulin - Novolog,inputevents,Medications,units,Solution,NaN,NaN,\binsulin\b,False
324,previous_insulin_treatment,223258,Insulin - Regular,Insulin - Regular,inputevents,Medications,units,Solution,NaN,NaN,\binsulin\b,False
3739,previous_insulin_treatment,229619,Insulin - U500,Insulin - U500,inputevents,Medications,units,Solution,NaN,NaN,\binsulin\b,False
1801,previous_insulin_treatment,226222,Insulin Ingredient,Insulin Ingredient,ingredientevents,Ingredients,units,Ingredient,NaN,NaN,\binsulin\b,False
2772,previous_insulin_treatment,228236,Insulin pump,Insulin pump,chartevents,Adm History/FHPA,NaN,Checkbox,NaN,NaN,\binsulin\b,False



previous_potassium_chloride_treatment: 4 inputevent item IDs


...


previous_norepinephrine_treatment: 1 inputevent item IDs


,variable,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue,patterns,exact_config_id
305,previous_norepinephrine_treatment,221906,Norepinephrine,Norepinephrine,inputevents,Medications,mg,Solution,NaN,NaN,"norepinephrine, noradrenaline",False



previous_fluids_given: 6 inputevent item IDs


,variable,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue,patterns,exact_config_id
1831,previous_fluids_given,226401,GU Irrigant - Normal Saline,GU Irrigant - Normal Saline,inputevents,Fluids/Intake,mL,Solution,NaN,NaN,"crystalloid, normal saline, lactated ringer, \...",False
1832,previous_fluids_given,226402,GU Irrigant - Sterile Water,GU Irrigant - Sterile Water,inputevents,Fluids/Intake,mL,Solution,NaN,NaN,"crystalloid, normal saline, lactated ringer, \...",False
1562,previous_fluids_given,225828,LR,LR,inputevents,Fluids/Intake,mL,Solution,NaN,NaN,"crystalloid, normal saline, lactated ringer, \...",False
1818,previous_fluids_given,226364,OR Crystalloid Intake,OR Crystalloid Intake,inputevents,Fluids/Intake,mL,Solution,NaN,NaN,"crystalloid, normal saline, lactated ringer, \...",False
1828,previous_fluids_given,226375,PACU Crystalloid Intake,PACU Crystalloid Intake,inputevents,Fluids/Intake,mL,Solution,NaN,NaN,"crystalloid, normal saline, lactated ringer, \...",False
1648,previous_fluids_given,225944,Sterile Water,Sterile Water,inputevents,Fluids/Intake,mL,Solution,NaN,NaN,"crystalloid, normal saline, lactated ringer, \...",False


,matched_input_itemids
variable,
previous_dextrose_treatment,18
previous_dopamine_treatment,1
previous_fluids_given,6
previous_insulin_treatment,10
previous_norepinephrine_treatment,1
previous_potassium_chloride_treatment,4


In [ ]:
chosen_treatment_items = {
    # ==========================================================
    # DEXTROSE
    # ==========================================================

    "dextrose_maintenance": [
        221017,  # Dextrose 2.5%
        220966,  # Dextrose 2.5% / Saline 0.45%
        220963,  # Dextrose 4.3% / Saline 0.18%
        220949,  # Dextrose 5%
        220964,  # Dextrose 5% / Saline 0.9%
        220965,  # Dextrose 5% / Saline 0.45%
        220967,  # Dextrose 5% / Ringer's lactate
        221000,  # Dextran 40 / Dextrose 5%
        221002,  # Dextran 70 / Dextrose 5%
    ],

    "dextrose_10": [
        220950,  # Dextrose 10%
        220968,  # Dextrose 10% / Ringer's lactate
    ],

    "dextrose_hypertonic": [
        221014,  # Dextrose 19%
        228140,  # Dextrose 20%
        220951,  # Dextrose 20%
        228141,  # Dextrose 30%
        228142,  # Dextrose 40%
        220952,  # Dextrose 50%
    ],


    # ==========================================================
    # INSULIN
    # ==========================================================

    # Rapid-acting analogues and regular insulin are combined because
    # all can affect glucose over the near-term prediction window.
    "insulin_prandial_short_acting": [
        223262,  # Humalog / lispro
        229299,  # Novolog / aspart
        223258,  # Regular insulin
    ],

    "insulin_intermediate": [
        223259,  # NPH
    ],

    "insulin_basal_long_acting": [
        223260,  # Glargine
    ],

    "insulin_premixed": [
        223257,  # Insulin 70/30
        223261,  # Humalog 75/25
    ],

    "insulin_u500": [
        229619,  # U-500 insulin
    ],

    # Dropped:
    # 226222 - Insulin Ingredient: ingredient-level record, likely duplicate
    # 228236 - Insulin pump: checkbox/history, not an administration event

    # ==========================================================
    # POTASSIUM CHLORIDE
    # ==========================================================

    "potassium_iv": [
        225166,  # Potassium Chloride
    ],

    "potassium_bolus": [
        227522,  # KCl bolus, inputevents
        # 227521 excluded because it is a chartevents entry rather than
        # a confirmed inputevents administration.
    ],

    "potassium_crrt": [
        227536,  # KCl during CRRT
    ],

    # ==========================================================
    # VASOPRESSORS
    # ==========================================================

    "dopamine": [
        221662,
    ],

    "norepinephrine": [
        221906,
    ],

    # ==========================================================
    # FLUIDS
    # ==========================================================

    "lactated_ringers": [
        225828,
    ]

}

## Write Selected Variables To Config

The dictionaries below are the chosen variables for each target. Running the next cell updates `configs/datasets/mimic/targets.yaml` so the adapter builds each target dataset from its own variable mapping.


In [ ]:
import copy
import yaml

SELECTED_LAB_ITEMIDS = copy.deepcopy(chosen_lab_itemids)
SELECTED_CHART_ITEMIDS = {
    'heart_rate': chosen_chart_itemids['heart_rate'],
    'systolic_bp': chosen_chart_itemids['systolic_bp'],
    'diastolic_bp': chosen_chart_itemids['diastolic_bp'],
    'o2_saturation_pulseox': chosen_chart_itemids['o2_saturation_pulseox'],
    'temperature_c': chosen_chart_itemids['temperature_c'] ,
    'temperature_f': chosen_chart_itemids['temperature_f']
}
SELECTED_INPUTEVENT_ITEMIDS = copy.deepcopy(chosen_treatment_items)

TARGET_VARIABLES = {
    'hypoglycemia': {
        'lab_itemids': SELECTED_LAB_ITEMIDS,
        'chart_itemids': SELECTED_CHART_ITEMIDS,
        'inputevent_itemids': {
            key: SELECTED_INPUTEVENT_ITEMIDS[key]
            for key in [
                'dextrose_maintenance',
                'dextrose_10',
                'dextrose_hypertonic',
                'insulin_prandial_short_acting',
                'insulin_intermediate',
                'insulin_basal_long_acting',
                'insulin_premixed',
                'insulin_u500',
                'lactated_ringers',
            ]
        },
    },
    'hypokalemia': {
        'lab_itemids': SELECTED_LAB_ITEMIDS,
        'chart_itemids': SELECTED_CHART_ITEMIDS,
        'inputevent_itemids': {
            key: SELECTED_INPUTEVENT_ITEMIDS[key]
            for key in ['potassium_iv', 'potassium_bolus', 'potassium_crrt', 'lactated_ringers']
        },
    },
    'hypotension': {
        'lab_itemids': SELECTED_LAB_ITEMIDS,
        'chart_itemids': SELECTED_CHART_ITEMIDS,
        'inputevent_itemids': {
            key: SELECTED_INPUTEVENT_ITEMIDS[key]
            for key in ['dopamine', 'norepinephrine', 'lactated_ringers']
        },
    },
}


def _plain_int_mapping(mapping):
    return {str(variable): [int(itemid) for itemid in itemids] for variable, itemids in mapping.items()}


def _plain_target_variables(target_variables):
    cleaned = {}
    for target, sections in target_variables.items():
        cleaned[str(target)] = {
            'lab_itemids': _plain_int_mapping(sections.get('lab_itemids', {})),
            'chart_itemids': _plain_int_mapping(sections.get('chart_itemids', {})),
            'inputevent_itemids': _plain_int_mapping(sections.get('inputevent_itemids', {})),
        }
    return cleaned


def write_target_variables_to_config(config_path, target_variables, *, targets=None):
    config_path = Path(config_path)
    with config_path.open('r', encoding='utf-8') as fh:
        raw = yaml.safe_load(fh) or {}

    cleaned = _plain_target_variables(target_variables)
    raw['targets'] = list(targets or cleaned.keys())
    raw['target_variables'] = cleaned

    # Keep global mapping fields empty so all variable selection is target-specific.
    raw['lab_itemids'] = {}
    raw['chart_itemids'] = {}
    raw['inputevent_itemids'] = {}
    raw.pop('lab_regexes', None)
    raw.pop('chart_regexes', None)
    raw.pop('inputevent_regexes', None)
    raw.pop('drug_regexes', None)

    with config_path.open('w', encoding='utf-8') as fh:
        yaml.safe_dump(raw, fh, sort_keys=False, default_flow_style=False)
    return config_path

written_config = write_target_variables_to_config(CONFIG_PATH, TARGET_VARIABLES)
print('Updated target variable mappings in', written_config)
TARGET_VARIABLES
